# EpiScope Google Colab Runner

This notebook runs the current EpiScope repository on Google Colab. The assignment is not yet fully complete: incremental hidden-drop processing, complete forecast evaluation, all required dashboard views, and final handover artifacts still need work.

Run cells from top to bottom. Do not store the hash salt in the notebook or repository.

## 1. Repository setup

Upload the repository to `/content/EpiScore`, or uncomment the clone command below.

In [ ]:
# Uncomment if cloning instead of uploading the repository.
# !git clone https://github.com/mdrobiulhassan/EpiScore.git /content/EpiScore

from pathlib import Path
import getpass
import os
import subprocess
import sys

REPO_DIR = Path('/content/EpiScore')
if not REPO_DIR.exists():
    raise FileNotFoundError('Upload or clone the repository at /content/EpiScore first.')
os.chdir(REPO_DIR)
print(Path.cwd())

## 2. Install pinned dependencies

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
# Spark workers must use the active Colab Python interpreter.
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Enter this at runtime; never commit it.
os.environ['EPISCOPE_HASH_SALT'] = getpass.getpass('EPISCOPE_HASH_SALT: ')
print(f'Python: {sys.executable}')

## 3. Verify Spark and PuLP

In [ ]:
subprocess.run([sys.executable, 'scripts/check_spark.py'], check=True)
subprocess.run([sys.executable, '-c', 'import pyspark, pulp; print(pyspark.__version__, pulp.__version__)'], check=True)

## 4. Smoke run

This validates generator, Bronze, Silver, Gold, analytics, serving, and dashboard export with 10,000 records.

In [ ]:
subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--rows', '10000', '--seed', '42'], check=True)

## 5. Assignment-scale run

The specification requires at least 25,000,000 encounter records at SF=1. This may require substantial Colab storage and runtime.

In [ ]:
subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--rows', '25000000', '--seed', '42'], check=True)

## 6. Scaling experiment

Archive the output metrics after each run if you need a reproducible comparison.

In [ ]:
for rows in (6250000, 12500000, 25000000):
    print(f'Running scale: {rows:,} rows')
    subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--rows', str(rows), '--seed', '42'], check=True)

## 7. Idempotency demonstration

Run the same input twice and compare Silver/Gold counts and duplicate checks.

In [ ]:
for _ in range(2):
    subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--rows', '10000', '--seed', '42'], check=True)

## 8. Hidden incremental drop

The repository does not currently implement a dedicated `--incremental-dir` argument. Do not run the command below until that entry point is implemented for the instructor's 14-day drop.

```python
# subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--incremental-dir', 'data/incremental_drop'], check=True)
```

## 9. Tests

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)

## 10. Dashboard payload check

In [ ]:
dashboard_files = sorted(Path('dashboard/data').glob('*.json'))
total_bytes = sum(path.stat().st_size for path in dashboard_files)
for path in dashboard_files:
    print(f'{path}: {path.stat().st_size / 1024 / 1024:.2f} MB')
print(f'Total: {total_bytes / 1024 / 1024:.2f} MB')
assert total_bytes <= 10 * 1024 * 1024, 'VIS-4 failed: payload exceeds 10 MB'

## 11. Serve the dashboard

In [ ]:
# Start the server in a background process.
dashboard_process = subprocess.Popen([sys.executable, '-m', 'http.server', '8000', '--directory', 'dashboard'])
print('Dashboard server started on port 8000')

In [ ]:
# Colab-only port forwarding.
from google.colab import output
output.serve_kernel_port_as_window(8000)